In [ ]:
# Function objects – functions are values in Python

def greet():
    print("Hello")

message = greet
message()

# Notice something important:
# - greet means the function object itself.
# - greet() means execute the function.
# This distinction is fundamental.

Hello


In [3]:
def calculate_profit(revenue, cost):
    return revenue - cost

profit_calculator = calculate_profit
print(profit_calculator(100000, 70000))

30000


In [5]:
print(calculate_profit is profit_calculator)
# Both names refer to the same function object.

True


In [6]:
# Functions can be stored in dictionaries
# Imagine a transaction processing engine. You support multiple transaction types.
def process_payment(transaction):
    return f"Processing payment: {transaction}"

def process_refund(transaction):
    return f"Processing refund: {transaction}"

def process_transfer(transaction):
    return f"Processing transfer: {transaction}"


transaction_handlers = {
    "PAYMENT": process_payment,
    "REFUND": process_refund,
    "TRANSFER": process_transfer
}

transaction_type = "REFUND"
transaction_id = 102

handler = transaction_handlers[transaction_type]
result = handler(transaction_id)

print(result)

Processing refund: 102


In [ ]:
# Higher-order functions
# Here's where things get interesting.

# A higher-order function is a function that does at least one of these:
# 1. Accepts another function as an argument.
# 2. Returns a function.

# 1. Passing a function as an argument
def add(a, b):
    return a + b

def multiply(a, b):
    return a * b

def calculate(a, b, operation):
    return operation(a, b)

print(calculate(10, 20, add))
print(calculate(10, 20, multiply))

# What happened?

# - Passed a function into another function.

# - Did not pass the result of a function.

## calculate(10, 20, add)    # Correct
## calculate(10, 20, add())  # Incorrect for this example

# add is the function object.
# add() attempts to execute it immediately, without the two required arguments.


30
200


### Callbacks – let another function decide what happens next
A callback is a function passed to another function so that it can be invoked at an appropriate point.

Let's use a business example.

Suppose you are processing a batch of transactions and want to perform a different action when each transaction is processed.

In [13]:
def on_success(transaction):
    print(f"Transaction {transaction['id']} processed successfully")

def on_failure(transaction):
    print(f"Transaction {transaction['id']} failed")

def process_transaction(transaction, callback):
    if transaction["amount"] > 0:
        callback(transaction)
    else:
        print("Invalid transaction amount")

transaction = {
    "id": 101,
    "amount": 5000,
}


process_transaction(transaction, on_success)

# Here, on_success is the callback.
# The processing function doesn't need to know all the details of what should happen after success. It delegates that decision to the callback.

Transaction 101 processed successfully


In [ ]:
# Returning functions from functions
# functions can be passed into other functions. Python also allows a function to return another function.
def create_greeter():
    def greet():
        print("Hello")

    return greet

my_greeter = create_greeter()

my_greeter()

# 1. create_greeter() executes.
# 2. Inside it, Python creates the function greet.
# 3. create_greeter returns the function object.
# 4. my_greeter refers to that function.
# 5. my_greeter() executes it.

Hello


In [15]:
# Returning customized functions
def create_multiplier(factor):

    def multiply(number):
        return number * factor

    return multiply

double = create_multiplier(2)
triple = create_multiplier(3)

print(double(10))
print(triple(10))


20
30


In [16]:
# example: configurable data validation
# Validate a transaction amount.
# Apply different business rules.
# Execute a callback when validation succeeds.
# Executing another callback when validation fails.

def is_positive_amount(amount):
    return amount > 0

def is_within_limit(amount, *, limit=100000):
    return amount <= limit

def validate_transaction(
        transaction,
        validator,
        *,
        on_success=None,
        on_failure=None
):
    amount = transaction["amount"]

    if validator(amount):
        if on_success:
            on_success(transaction)
        return True

    if on_failure:
        on_failure(transaction)

    return False


def success_handler(transaction):
    print(f"Accepted transaction: {transaction['id']}")


def failure_handler(transaction):
    print(f"Rejected transaction: {transaction['id']}")

validate_transaction(
    transaction,
    is_positive_amount,
    on_success=success_handler,
    on_failure=failure_handler
)

Accepted transaction: 101


True